# NB02：水分與水活性（Moisture Content and Water Activity）

**食品分析實驗 週次 2｜Nielsen's Food Analysis, 6th ed. 對應章節：水分分析（Moisture and Total Solids Analysis）、水活性**

## 學習目標 (Learning Objectives)

完成本筆記本後，你應該能夠：

1. 由坩堝(鋁盤)、樣品、乾燥後三次秤重數據，計算**濕基(wet basis)** 與 **乾基(dry basis)** 水分含量 %。
2. 應用「連續兩次秤重差 ≤ 2 mg」的**恆重(constant weight)** 判準，判斷是否需要再烘一次。
3. 計算各組平均值、SD、RSD%，並用**成對比較(paired comparison)** 評估烘箱法與紅外線水分計的偏差(bias)。
4. 用散佈圖說明「水分％相同 ≠ 水活性 a<sub>w</sub> 相同」，並連結到微生物生長下限。
5. 分析全學期共同盲樣（全脂奶粉）的班級數據 (n=5 組)。
6. 用質量平衡(mass balance)檢驗牛肉乾加工產率的行銷說法是否合理。

## 如何使用本筆記本

- 直接「全部執行」（Run all）即可用內建的**模擬教學資料**跑完整份筆記本。
- `load_data()` 支援三種資料來源，用法與 NB01 相同：
  1. 本機/雲端硬碟路徑：`load_data("data/nb02_sample.csv")`
  2. 已發布的 Google 試算表 CSV 連結：`load_data("https://docs.google.com/.../pub?output=csv")`
  3. 在 Colab 上傳檔案（Setup 區塊有註解掉的程式碼）
  4. 都沒有提供時，自動退回本筆記本內嵌的模擬資料字串。

> ⚠️ **本筆記本內建的資料是模擬教學資料（simulated teaching data）**，不是真實量測值。做作業時請換成你們班的實際數據。

## 資料格式 (Data Schema)

沿用 NB01 的長格式，本週多加一個欄位 `sample`（食品樣品名稱，因為本週同一位學生會測不只一種食品/一種量測項目）：

| 欄位 | 說明 |
|---|---|
| `group` | 組別（1–5） |
| `student` | 學號/代號 |
| `item` | 量測項目代號（見下方各節說明，例如 `oven_wet_g`、`ir_moisture_pct`、`aw_reading`） |
| `rep` | 重複次數 |
| `value` | 量測值 |
| `unit` | `value` 的單位（`g`、`%`、`aw`） |
| `temp_C` | 量測當下的溫度（°C）——烘箱溫度或環境溫度，依項目而定 |
| `note` | 備註欄（如恆重判定說明） |
| `sample` | **（本週新增）** 食品樣品名稱，例如 `beef_jerky`、`milk_powder` |

## 1. 環境設定 (Setup)

匯入套件並定義 `load_data()`。所有圖表座標軸與標題一律使用**英文**，避免中文字型在 Colab 顯示為方框（tofu）。

In [ ]:
import io
import os

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

plt.style.use("ggplot")
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.precision", 4)

RNG_SEED = 42
np.random.seed(RNG_SEED)

In [ ]:
# ---- Embedded sample data (fallback) ----------------------------------
# Exact content of data/nb02_sample.csv, embedded so the notebook is
# self-contained (works even without the data/ folder present).
SAMPLE_CSV = """group,student,item,rep,value,unit,temp_C,note,sample
1,G1_S1,oven_dish_g,1,18.8698,g,103.4,,beef_jerky
1,G1_S1,oven_wet_g,1,21.8178,g,103.4,,beef_jerky
1,G1_S1,oven_dry1_g,1,21.2171,g,103.4,,beef_jerky
1,G1_S1,oven_dry2_g,1,21.2158,g,103.4,第2次與第1次秤重差 ≤ 2 mg，達恆重,beef_jerky
1,G1_S1,ir_moisture_pct,1,19.63,%,25.5,,beef_jerky
1,G1_S2,oven_dish_g,1,18.9303,g,105.6,,beef_jerky
1,G1_S2,oven_wet_g,1,21.9295,g,105.6,,beef_jerky
1,G1_S2,oven_dry1_g,1,21.3425,g,105.6,,beef_jerky
1,G1_S2,oven_dry2_g,1,21.3418,g,105.6,第2次與第1次秤重差 ≤ 2 mg，達恆重,beef_jerky
1,G1_S2,ir_moisture_pct,1,19.22,%,25.6,,beef_jerky
1,G1_S3,oven_dish_g,1,17.2171,g,105.7,,beef_jerky
1,G1_S3,oven_wet_g,1,20.1741,g,105.7,,beef_jerky
1,G1_S3,oven_dry1_g,1,19.5772,g,105.7,,beef_jerky
1,G1_S3,oven_dry2_g,1,19.577,g,105.7,第2次與第1次秤重差 ≤ 2 mg，達恆重,beef_jerky
1,G1_S3,ir_moisture_pct,1,19.78,%,25.5,,beef_jerky
2,G2_S1,oven_dish_g,1,16.7726,g,104.7,,dried_longan
2,G2_S1,oven_wet_g,1,19.8337,g,104.7,,dried_longan
2,G2_S1,oven_dry1_g,1,19.2238,g,104.7,,dried_longan
2,G2_S1,oven_dry2_g,1,19.2224,g,104.7,第2次與第1次秤重差 ≤ 2 mg，達恆重,dried_longan
2,G2_S1,ir_moisture_pct,1,19.73,%,24.1,,dried_longan
2,G2_S2,oven_dish_g,1,15.7714,g,104.6,,dried_longan
2,G2_S2,oven_wet_g,1,18.7929,g,104.6,,dried_longan
2,G2_S2,oven_dry1_g,1,18.1562,g,104.6,,dried_longan
2,G2_S2,oven_dry2_g,1,18.1545,g,104.6,第2次與第1次秤重差 ≤ 2 mg，達恆重,dried_longan
2,G2_S2,ir_moisture_pct,1,20.48,%,24.9,,dried_longan
2,G2_S3,oven_dish_g,1,15.9474,g,105.5,,dried_longan
2,G2_S3,oven_wet_g,1,18.9417,g,105.5,,dried_longan
2,G2_S3,oven_dry1_g,1,18.3554,g,105.5,,dried_longan
2,G2_S3,oven_dry2_g,1,18.3549,g,105.5,第2次與第1次秤重差 ≤ 2 mg，達恆重,dried_longan
2,G2_S3,ir_moisture_pct,1,19.42,%,25.7,,dried_longan
3,G3_S1,oven_dish_g,1,18.5013,g,105.1,,white_bread
3,G3_S1,oven_wet_g,1,21.5285,g,105.1,,white_bread
3,G3_S1,oven_dry1_g,1,20.4791,g,105.1,,white_bread
3,G3_S1,oven_dry2_g,1,20.4776,g,105.1,第2次與第1次秤重差 ≤ 2 mg，達恆重,white_bread
3,G3_S1,ir_moisture_pct,1,34.38,%,25.4,,white_bread
3,G3_S2,oven_dish_g,1,15.6988,g,103.8,,white_bread
3,G3_S2,oven_wet_g,1,18.7327,g,103.8,,white_bread
3,G3_S2,oven_dry1_g,1,17.6698,g,103.8,,white_bread
3,G3_S2,oven_dry2_g,1,17.6634,g,103.8,第2次與第1次秤重差 > 2 mg，尚未恆重，需再烘1小時,white_bread
3,G3_S2,oven_dry3_g,1,17.6623,g,103.8,第3次與第2次秤重差 ≤ 2 mg，達恆重,white_bread
3,G3_S2,ir_moisture_pct,1,34.79,%,24.9,,white_bread
3,G3_S3,oven_dish_g,1,17.8437,g,105.8,,white_bread
3,G3_S3,oven_wet_g,1,20.8299,g,105.8,,white_bread
3,G3_S3,oven_dry1_g,1,19.7624,g,105.8,,white_bread
3,G3_S3,oven_dry2_g,1,19.7612,g,105.8,第2次與第1次秤重差 ≤ 2 mg，達恆重,white_bread
3,G3_S3,ir_moisture_pct,1,34.88,%,25.5,,white_bread
4,G4_S1,oven_dish_g,1,18.1736,g,104.7,,potato_chips
4,G4_S1,oven_wet_g,1,21.2029,g,104.7,,potato_chips
4,G4_S1,oven_dry1_g,1,21.1315,g,104.7,,potato_chips
4,G4_S1,oven_dry2_g,1,21.1309,g,104.7,第2次與第1次秤重差 ≤ 2 mg，達恆重,potato_chips
4,G4_S1,ir_moisture_pct,1,1.84,%,24.4,,potato_chips
4,G4_S2,oven_dish_g,1,17.0426,g,105.4,,potato_chips
4,G4_S2,oven_wet_g,1,19.9788,g,105.4,,potato_chips
4,G4_S2,oven_dry1_g,1,19.9367,g,105.4,,potato_chips
4,G4_S2,oven_dry2_g,1,19.9365,g,105.4,第2次與第1次秤重差 ≤ 2 mg，達恆重,potato_chips
4,G4_S2,ir_moisture_pct,1,1.08,%,25.3,,potato_chips
4,G4_S3,oven_dish_g,1,17.7852,g,105.4,,potato_chips
4,G4_S3,oven_wet_g,1,20.7931,g,105.4,,potato_chips
4,G4_S3,oven_dry1_g,1,20.7235,g,105.4,,potato_chips
4,G4_S3,oven_dry2_g,1,20.7227,g,105.4,第2次與第1次秤重差 ≤ 2 mg，達恆重,potato_chips
4,G4_S3,ir_moisture_pct,1,1.74,%,24.0,,potato_chips
5,G5_S1,oven_dish_g,1,15.4502,g,105.0,,milk_powder
5,G5_S1,oven_wet_g,1,18.3904,g,105.0,,milk_powder
5,G5_S1,oven_dry1_g,1,18.295,g,105.0,,milk_powder
5,G5_S1,oven_dry2_g,1,18.2946,g,105.0,第2次與第1次秤重差 ≤ 2 mg，達恆重,milk_powder
5,G5_S1,ir_moisture_pct,1,3.0,%,25.4,,milk_powder
5,G5_S2,oven_dish_g,1,17.2308,g,103.7,,milk_powder
5,G5_S2,oven_wet_g,1,20.2259,g,103.7,,milk_powder
5,G5_S2,oven_dry1_g,1,20.1424,g,103.7,,milk_powder
5,G5_S2,oven_dry2_g,1,20.1412,g,103.7,第2次與第1次秤重差 ≤ 2 mg，達恆重,milk_powder
5,G5_S2,ir_moisture_pct,1,1.99,%,24.2,,milk_powder
5,G5_S3,oven_dish_g,1,19.8095,g,106.0,,milk_powder
5,G5_S3,oven_wet_g,1,22.8295,g,106.0,,milk_powder
5,G5_S3,oven_dry1_g,1,22.7526,g,106.0,,milk_powder
5,G5_S3,oven_dry2_g,1,22.752,g,106.0,第2次與第1次秤重差 ≤ 2 mg，達恆重,milk_powder
5,G5_S3,ir_moisture_pct,1,2.06,%,25.4,,milk_powder
1,G1_S1,moisture_pct_survey,1,19.07,%,24.2,,beef_jerky
1,G1_S1,aw_reading,1,0.747,aw,24.2,,beef_jerky
1,G1_S1,moisture_pct_survey,1,19.66,%,24.4,,dried_longan
1,G1_S1,aw_reading,1,0.613,aw,24.4,,dried_longan
1,G1_S1,moisture_pct_survey,1,35.76,%,24.4,,white_bread
1,G1_S1,aw_reading,1,0.954,aw,24.4,,white_bread
1,G1_S1,moisture_pct_survey,1,1.28,%,25.4,,potato_chips
1,G1_S1,aw_reading,1,0.201,aw,25.4,,potato_chips
1,G1_S1,moisture_pct_survey,1,3.12,%,25.2,,milk_powder
1,G1_S1,aw_reading,1,0.25,aw,25.2,,milk_powder
1,G1_S1,moisture_pct_survey,1,16.8,%,24.8,蜂蜜以紅外線水分計量測(不適用常壓烘箱),honey
1,G1_S1,aw_reading,1,0.585,aw,24.8,,honey
2,G2_S1,moisture_pct_survey,1,20.22,%,24.7,,beef_jerky
2,G2_S1,aw_reading,1,0.77,aw,24.7,,beef_jerky
2,G2_S1,moisture_pct_survey,1,20.36,%,25.2,,dried_longan
2,G2_S1,aw_reading,1,0.622,aw,25.2,,dried_longan
2,G2_S1,moisture_pct_survey,1,33.88,%,25.2,,white_bread
2,G2_S1,aw_reading,1,0.936,aw,25.2,,white_bread
2,G2_S1,moisture_pct_survey,1,1.31,%,24.0,,potato_chips
2,G2_S1,aw_reading,1,0.21,aw,24.0,,potato_chips
2,G2_S1,moisture_pct_survey,1,2.26,%,25.6,,milk_powder
2,G2_S1,aw_reading,1,0.235,aw,25.6,,milk_powder
2,G2_S1,moisture_pct_survey,1,17.71,%,25.0,蜂蜜以紅外線水分計量測(不適用常壓烘箱),honey
2,G2_S1,aw_reading,1,0.63,aw,25.0,,honey
3,G3_S1,moisture_pct_survey,1,20.41,%,24.9,,beef_jerky
3,G3_S1,aw_reading,1,0.735,aw,24.9,,beef_jerky
3,G3_S1,moisture_pct_survey,1,20.27,%,25.0,,dried_longan
3,G3_S1,aw_reading,1,0.588,aw,25.0,,dried_longan
3,G3_S1,moisture_pct_survey,1,33.93,%,25.7,,white_bread
3,G3_S1,aw_reading,1,0.948,aw,25.7,,white_bread
3,G3_S1,moisture_pct_survey,1,2.08,%,25.1,,potato_chips
3,G3_S1,aw_reading,1,0.198,aw,25.1,,potato_chips
3,G3_S1,moisture_pct_survey,1,2.16,%,24.6,,milk_powder
3,G3_S1,aw_reading,1,0.243,aw,24.6,,milk_powder
3,G3_S1,moisture_pct_survey,1,17.24,%,25.5,蜂蜜以紅外線水分計量測(不適用常壓烘箱),honey
3,G3_S1,aw_reading,1,0.627,aw,25.5,,honey
4,G4_S1,moisture_pct_survey,1,20.98,%,24.5,,beef_jerky
4,G4_S1,aw_reading,1,0.743,aw,24.5,,beef_jerky
4,G4_S1,moisture_pct_survey,1,19.03,%,24.7,,dried_longan
4,G4_S1,aw_reading,1,0.589,aw,24.7,,dried_longan
4,G4_S1,moisture_pct_survey,1,33.56,%,24.6,,white_bread
4,G4_S1,aw_reading,1,0.963,aw,24.6,,white_bread
4,G4_S1,moisture_pct_survey,1,2.47,%,25.0,,potato_chips
4,G4_S1,aw_reading,1,0.206,aw,25.0,,potato_chips
4,G4_S1,moisture_pct_survey,1,2.98,%,24.3,,milk_powder
4,G4_S1,aw_reading,1,0.24,aw,24.3,,milk_powder
4,G4_S1,moisture_pct_survey,1,16.61,%,26.0,蜂蜜以紅外線水分計量測(不適用常壓烘箱),honey
4,G4_S1,aw_reading,1,0.582,aw,26.0,,honey
5,G5_S1,moisture_pct_survey,1,20.17,%,25.8,,beef_jerky
5,G5_S1,aw_reading,1,0.774,aw,25.8,,beef_jerky
5,G5_S1,moisture_pct_survey,1,19.88,%,24.6,,dried_longan
5,G5_S1,aw_reading,1,0.604,aw,24.6,,dried_longan
5,G5_S1,moisture_pct_survey,1,35.38,%,24.7,,white_bread
5,G5_S1,aw_reading,1,0.944,aw,24.7,,white_bread
5,G5_S1,moisture_pct_survey,1,2.21,%,24.5,,potato_chips
5,G5_S1,aw_reading,1,0.223,aw,24.5,,potato_chips
5,G5_S1,moisture_pct_survey,1,2.39,%,24.2,,milk_powder
5,G5_S1,aw_reading,1,0.229,aw,24.2,,milk_powder
5,G5_S1,moisture_pct_survey,1,18.47,%,24.4,蜂蜜以紅外線水分計量測(不適用常壓烘箱),honey
5,G5_S1,aw_reading,1,0.597,aw,24.4,,honey
"""

In [ ]:
def load_data(source=None):
    """Load NB02 moisture/aw data in the shared long-format schema:
    group, student, item, rep, value, unit, temp_C, note, sample

    Parameters
    ----------
    source : str or None
        - None: try "data/nb02_sample.csv" first, else the embedded fallback.
        - a local file path: read with pandas.read_csv.
        - an http(s) URL (e.g. a published Google Sheets CSV link).

    Returns
    -------
    pandas.DataFrame
    """
    if source is not None:
        try:
            return pd.read_csv(source)
        except Exception as e:
            print(f"讀取 {source} 失敗，改用內建模擬資料。錯誤訊息：{e}")

    local_path = os.path.join("data", "nb02_sample.csv")
    if os.path.exists(local_path):
        return pd.read_csv(local_path)

    return pd.read_csv(io.StringIO(SAMPLE_CSV))


df = load_data()
print(f"讀入 {len(df)} 筆資料，共 {df['student'].nunique()} 位學生、"
      f"{df['group'].nunique()} 組、{df['item'].nunique()} 個量測項目、"
      f"{df['sample'].nunique()} 種食品樣品。")
df.head()

### （選用）在 Colab 上傳班級實際資料

In [ ]:
# try:
#     from google.colab import files
#     uploaded = files.upload()
#     fname = list(uploaded.keys())[0]
#     df = load_data(fname)
#     print(f"已改用上傳檔案：{fname}")
# except ImportError:
#     print("非 Colab 環境，略過上傳步驟；請改用 load_data('本機路徑或URL')")

## 2. 烘箱法水分含量：濕基 vs 乾基、恆重判斷

### 公式

$$濕基(wet\ basis)\% = \frac{W_{濕} - W_{乾}}{W_{濕}} \times 100 \qquad
乾基(dry\ basis)\% = \frac{W_{濕} - W_{乾}}{W_{乾}} \times 100$$

（$W_{濕}$、$W_{乾}$ 都是「樣品淨重」＝盤+樣品重 − 空盤重）

### 範例（沿用課程講義數字）

空盤重 **25.1234 g**；盤+樣品 **28.1234 g**；乾燥後(盤+乾樣品) **27.9734 g**：

- 樣品重 = 28.1234 − 25.1234 = **3.0000 g**
- 水分重 = 28.1234 − 27.9734 = **0.1500 g**
- 乾固形物重 = 3.0000 − 0.1500 = **2.8500 g**
- 濕基 % = 0.1500 / 3.0000 × 100 = **5.00%**
- 乾基 % = 0.1500 / 2.8500 × 100 = **5.26%**

兩者分母不同：濕基除以「原始濕重」，乾基除以「乾固形物重」。

### 恆重 (Constant Weight) 判準

連續兩次乾燥後秤重，**差值 ≤ 2 mg（0.002 g）** 才算恆重；否則要回烘箱再烘（本課程 SOP，實務依實驗室規定可能不同）。本週每組 3 位同學各自完成一次完整烘箱測定（等於組內三重複），其中一位同學的數據會需要**第三次秤重**才達恆重——這是真實實驗常見的情況，不是資料錯誤。

In [ ]:
def wet_basis_pct(wet_g, dry_g, dish_g):
    return (wet_g - dry_g) / (wet_g - dish_g) * 100.0


def dry_basis_pct(wet_g, dry_g, dish_g):
    return (wet_g - dry_g) / (dry_g - dish_g) * 100.0


# Worked example from the lecture notes
V_wet, V_dry, V_dish = 28.1234, 27.9734, 25.1234
print(f"濕基 % = {wet_basis_pct(V_wet, V_dry, V_dish):.2f}%  (expect 5.00%)")
print(f"乾基 % = {dry_basis_pct(V_wet, V_dry, V_dish):.2f}%  (expect 5.26%)")

In [ ]:
CONSTANT_WEIGHT_TOL_G = 0.002  # 2 mg


def resolve_final_dry_mass(row):
    """Apply the constant-weight rule: use dry2 if |dry2-dry1| <= 2 mg,
    otherwise dry3 (only present when a 3rd weighing was needed).
    Returns (mass, weighing used, last difference in mg, constant weight reached?)."""
    tol_mg = CONSTANT_WEIGHT_TOL_G * 1000
    diff12_mg = abs(row["oven_dry2_g"] - row["oven_dry1_g"]) * 1000
    if diff12_mg <= tol_mg:
        return row["oven_dry2_g"], 2, diff12_mg, True
    if not pd.isna(row.get("oven_dry3_g", np.nan)):
        diff23_mg = abs(row["oven_dry3_g"] - row["oven_dry2_g"]) * 1000
        # dry3 counts only if it also agrees with dry2 within 2 mg
        return row["oven_dry3_g"], 3, diff23_mg, diff23_mg <= tol_mg
    return row["oven_dry2_g"], 2, diff12_mg, False  # not constant, no 3rd weighing recorded


oven_items = ["oven_dish_g", "oven_wet_g", "oven_dry1_g", "oven_dry2_g", "oven_dry3_g"]
oven_wide = (
    df[df["item"].isin(oven_items)]
    .pivot_table(index=["group", "student", "sample"], columns="item", values="value", aggfunc="first")
    .reset_index()
)

final_mass, weighing_used, final_diff_mg, reached = [], [], [], []
for _, r in oven_wide.iterrows():
    m, n_weigh, d, ok = resolve_final_dry_mass(r)
    final_mass.append(m)
    weighing_used.append(n_weigh)
    final_diff_mg.append(d)
    reached.append(ok)

oven_wide["final_dry_g"] = final_mass
oven_wide["weighings_used"] = weighing_used
oven_wide["final_diff_mg"] = final_diff_mg
oven_wide["constant_weight"] = reached
oven_wide["wet_basis_pct"] = wet_basis_pct(oven_wide["oven_wet_g"], oven_wide["final_dry_g"], oven_wide["oven_dish_g"])
oven_wide["dry_basis_pct"] = dry_basis_pct(oven_wide["oven_wet_g"], oven_wide["final_dry_g"], oven_wide["oven_dish_g"])

print("需要第三次秤重才達恆重的樣本：")
print(oven_wide[oven_wide["weighings_used"] == 3][["group", "student", "sample", "final_diff_mg"]])
not_constant = oven_wide[~oven_wide["constant_weight"]]
if len(not_constant):
    print("⚠ 以下樣本最後兩次秤重差仍 > 2 mg，尚未恆重，需再烘；水分值只能當暫定值：")
    print(not_constant[["group", "student", "sample", "weighings_used", "final_diff_mg"]])
else:
    print("所有樣本都已達恆重。")
oven_wide[["group", "student", "sample", "wet_basis_pct", "dry_basis_pct", "weighings_used"]]

## 3. 各組（各食品）平均、SD、RSD%

In [ ]:
group_stats = (
    oven_wide.groupby(["group", "sample"])["wet_basis_pct"]
    .agg(n="count", mean_pct="mean", sd_pct="std")
    .reset_index()
)
group_stats["rsd_pct"] = group_stats["sd_pct"] / group_stats["mean_pct"] * 100
group_stats

## 4. 烘箱法 vs 紅外線水分計：成對比較 (Paired Comparison)

同一批樣品（同一位學生、同一份食品）分別用烘箱法（參考法）與紅外線水分計量測，兩者的差異可用來評估紅外線水分計的**偏差(bias)**。這裡先用簡單的成對 t 檢定與差異圖做「初步線索」，**完整的方法比較統計（Bland–Altman、F 檢定等）會在 W4／NB04 更嚴謹地處理**。

In [ ]:
ir_df = df[df["item"] == "ir_moisture_pct"][["group", "student", "sample", "value"]].rename(
    columns={"value": "ir_pct"})
paired = oven_wide.merge(ir_df, on=["group", "student", "sample"])
paired["diff_pct"] = paired["ir_pct"] - paired["wet_basis_pct"]

bias = paired["diff_pct"].mean()
t_stat, p_value = stats.ttest_rel(paired["ir_pct"], paired["wet_basis_pct"])
print(f"平均偏差 (IR − 烘箱) = {bias:+.3f}%")
print(f"成對 t 檢定：t = {t_stat:.3f}, p = {p_value:.4f}")
if p_value < 0.05:
    print("p < 0.05 -> 兩法之間存在統計上顯著的系統性差異（bias）。")
else:
    print("p >= 0.05 -> 沒有足夠證據顯示兩法有系統性差異。")

paired[["group", "student", "sample", "wet_basis_pct", "ir_pct", "diff_pct"]]

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
mean_val = (paired["wet_basis_pct"] + paired["ir_pct"]) / 2
ax.scatter(mean_val, paired["diff_pct"], s=60, color="tab:blue")
ax.axhline(bias, color="tab:red", linewidth=1.5, label=f"mean bias = {bias:+.2f}%")
ax.axhline(0, color="grey", linewidth=1, linestyle="--")
for _, r in paired.iterrows():
    ax.annotate(r["sample"], ((r["wet_basis_pct"] + r["ir_pct"]) / 2, r["diff_pct"]),
                textcoords="offset points", xytext=(5, 4), fontsize=7)
ax.set_xlabel("Mean of oven and IR moisture (%)")
ax.set_ylabel("Difference: IR - oven (%)")
ax.set_title("Oven vs IR moisture: difference plot (preview of Bland-Altman, see NB04)")
ax.legend()
plt.tight_layout()
plt.show()

## 5. 水活性 a<sub>w</sub> vs 水分含量：同水分％不代表同 a<sub>w</sub>

下面用全班（5 組）對 6 種食品的水活性計 + 快速水分量測資料畫散佈圖。特別注意 **牛肉乾** 與 **龍眼乾** 水分含量都約 20%，但 a<sub>w</sub> 明顯不同（約 0.75 vs 0.60）——水分「量」相同不代表水的「有效性」相同。

**典型微生物生長下限（教材值）：** 細菌 a<sub>w</sub>≈0.91、酵母菌≈0.88、黴菌≈0.80、嗜旱性黴菌≈0.61。

In [ ]:
survey = df[df["item"].isin(["moisture_pct_survey", "aw_reading"])].pivot_table(
    index=["group", "sample"], columns="item", values="value", aggfunc="first").reset_index()

fig, ax = plt.subplots(figsize=(9, 6))
colors = plt.cm.tab10.colors
for i, food in enumerate(survey["sample"].unique()):
    sub = survey[survey["sample"] == food]
    ax.scatter(sub["moisture_pct_survey"], sub["aw_reading"], s=70, color=colors[i % 10], label=food)

for aw_line, label in [(0.91, "bacteria min"), (0.88, "yeast min"), (0.80, "mold min"), (0.61, "xerophilic mold min")]:
    ax.axhline(aw_line, color="grey", linestyle=":", linewidth=1)
    ax.text(0.5, aw_line + 0.01, label, fontsize=8, color="grey")

ax.set_xlabel("Moisture content (%, wet basis)")
ax.set_ylabel("Water activity (aw)")
ax.set_title("aw vs moisture across different foods (n=5 groups per food)")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

## 6. 全學期共同盲樣：全脂奶粉水分 (n = 5 組)

全脂奶粉是本課程**全學期共同盲樣**，之後 W3/W5/W6/W8 會用同一批樣品追蹤不同分析項目。這裡先看本週 5 組的水分測定結果（來自第 5 節的快速水分量測）是否一致。

In [ ]:
blind = survey[survey["sample"] == "milk_powder"]
n = len(blind)
mean_pct = blind["moisture_pct_survey"].mean()
sd_pct = blind["moisture_pct_survey"].std(ddof=1)
rsd_pct = sd_pct / mean_pct * 100
print(f"全脂奶粉共同盲樣 (n={n} 組)：mean = {mean_pct:.2f}%, SD = {sd_pct:.2f}%, RSD = {rsd_pct:.1f}%")
blind[["group", "moisture_pct_survey"]]

## 7. 練習題 (Exercises)

請在每題下方的空白程式碼區塊中作答。

### 問題 1：牛肉乾質量平衡 (Mass Balance)

牛肉乾工廠影片聲稱：**500 g 鮮肉 → 100 g 牛肉乾**。已知瘦牛肉約 **73% 水分**（即 27% 固形物），若成品牛肉乾約 **20% 水分**（80% 固形物），且加工過程中固形物幾乎不流失：

1. 用質量平衡計算合理的產率（%）與最終克數。
2. 這個產率跟影片聲稱的 100 g 差多少？
3. （額外思考）100 g 的產率(20%水分)是否還能同時支持影片「>30 g 蛋白質/100 g」的說法？

In [ ]:
# TODO: 請在這裡作答

### 問題 2：wet basis / dry basis 計算練習

某樣品：空盤重 **22.3300 g**；盤+樣品 **26.3300 g**；乾燥後(盤+乾樣品) **25.5300 g**。分別計算濕基%與乾基%，並說明為什麼兩個數字不一樣。

In [ ]:
# TODO: 請在這裡作答

### 問題 3：恆重判斷

某樣品連續三次乾燥後秤重為：12.4050 g → 12.3890 g → 12.3860 g。用「連續兩次差 ≤ 2 mg」的規則：

1. 第 3 次秤重後達到恆重了嗎？請算出每兩次之間的差（mg）。
2. 如果還沒，下一步該怎麼做？
3. 再烘一次後第 4 次秤得 12.3851 g，現在達到恆重了嗎？該用哪一個重量計算最終水分含量？


In [ ]:
# TODO: 請在這裡作答

### 問題 4：RSD% 比較

用 `group_stats`（第 3 節）比較不同食品的 RSD%，哪一種食品的精密度看起來最差？可能的操作或樣品本身原因是什麼？

In [ ]:
# TODO: 請在這裡作答

### 問題 5：同水分不同 a<sub>w</sub>

牛肉乾與龍眼乾水分都約 20%，但 a<sub>w</sub> 不同。哪一個更容易發黴？請用第 5 節圖中的微生物生長下限說明你的判斷。

In [ ]:
# TODO: 請在這裡作答

## 8. 匯出結果 (Export)

把各組食品的水分統計結果存成 CSV。

In [ ]:
output_path = "nb02_group_results.csv"
group_stats.to_csv(output_path, index=False)
print(f"已儲存：{output_path}")

try:
    from google.colab import files
    files.download(output_path)
except ImportError:
    print("非 Colab 環境，略過自動下載（檔案已存在本機/雲端硬碟工作目錄）。")